# Portfolio backtest (4 pairs)

Este notebook corre o backtest **holdout** de 4 pares (cada um com o seu ensemble Conv1D+LSTM+Hybrid, carregando os `.pt` na pasta do par) e depois cria uma **equity curve combinada** (portfolio) e calcula **max drawdown**.

- Ajusta `PAIRS` e os parâmetros por par (SL/TP/threshold/pos_size) na secção de configuração.
- Por default usa os mesmos hiperparâmetros do backtest atual; a ideia é poderes afinar **por par** mantendo o ensemble constante.


In [ ]:
import os
import re
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MaxAbsScaler

import torch
import torch.nn as nn


def set_all_seeds(seed: int = 0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)


def calculate_max_drawdown(equities):
    equity = np.array(equities, dtype=float)
    peak = np.maximum.accumulate(equity)
    drawdown = (peak - equity) / peak
    max_dd_idx = int(np.argmax(drawdown))
    peak_idx = int(np.argmax(equity[: max_dd_idx + 1]))
    return {
        "max_drawdown": float(drawdown[max_dd_idx]),
        "peak_index": peak_idx,
        "trough_index": max_dd_idx,
        "peak_value": float(equity[peak_idx]),
        "trough_value": float(equity[max_dd_idx]),
    }


def calculate_sharpe_ratio(equities):
    equity = np.array(equities, dtype=float)
    rets = (equity[1:] - equity[:-1]) / equity[:-1]
    if len(rets) < 2:
        return 0.0
    mean = float(np.mean(rets))
    std = float(np.std(rets, ddof=1))
    return round(mean / std, 4) if std > 0 else 0.0


def plot_equity_simple(equities, title: str = "Equity"):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(equities, lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("Time step")
    ax.set_ylabel("Equity")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
def build_features(opens, highs, lows, closes, volumes, train_scalers=None):
    # NOTE: This mirrors the backtest notebook logic.
    opens = np.asarray(opens)
    highs = np.asarray(highs)
    lows = np.asarray(lows)
    closes = np.asarray(closes)
    volumes = np.asarray(volumes)

    # Features
    r_close = np.diff(closes, prepend=closes[0]) / np.maximum(closes, 1e-12)
    r_open = np.diff(opens, prepend=opens[0]) / np.maximum(opens, 1e-12)
    hl_range = (highs - lows) / np.maximum(closes, 1e-12)
    vol = volumes

    features = np.stack([r_close, r_open, hl_range, vol], axis=1)
    num_features = features.shape[1]

    if train_scalers is None:
        train_scalers = [MaxAbsScaler() for _ in range(num_features)]
        for i in range(num_features):
            features[:, i : i + 1] = train_scalers[i].fit_transform(features[:, i : i + 1]).reshape(-1)
    else:
        for i in range(num_features):
            features[:, i : i + 1] = train_scalers[i].transform(features[:, i : i + 1]).reshape(-1)

    return features, num_features, train_scalers


def preprocess_data(seq_len: int, df: pd.DataFrame, train_scalers=None):
    opens = df["open"].to_numpy(dtype=float)
    highs = df["high"].to_numpy(dtype=float)
    lows = df["low"].to_numpy(dtype=float)
    closes = df["close"].to_numpy(dtype=float)
    volumes = df["volume"].to_numpy(dtype=float)

    features, num_features, train_scalers = build_features(opens, highs, lows, closes, volumes, train_scalers)

    X = []
    Y = []
    for i in range(seq_len, len(features)):
        X.append(features[i - seq_len : i])
        Y.append(features[i])

    X = np.asarray(X)
    Y = np.asarray(Y)

    # Align OHLC arrays to X/Y lengths
    opens = opens[seq_len:]
    highs = highs[seq_len:]
    lows = lows[seq_len:]
    closes = closes[seq_len:]

    return X, Y, num_features, opens, highs, lows, closes, train_scalers


class Model_1(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=3, padding=1, stride=1)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.act2 = nn.GELU()
        self.fc_out = nn.Linear(64, num_features)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act1(self.conv1(x))
        x = self.act2(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = x[:, -1, :]
        return self.fc_out(x)


class Model_2(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        self.lstm = nn.LSTM(input_size=num_features, hidden_size=64, num_layers=1, batch_first=True)
        self.fc_out = nn.Linear(64, num_features)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        return self.fc_out(x)


class Model_3(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, 64, kernel_size=3, padding=1, stride=1)
        self.act1 = nn.GELU()
        self.lstm = nn.LSTM(64, 32, batch_first=True)
        self.fc_out = nn.Linear(32, num_features)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act1(self.conv1(x))
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        return self.fc_out(x)


_EQ_RE = re.compile(r"^eq_(?P<eq>\d+)_ep_(?P<ep>\d+)\.pt$")


def top_k_checkpoints(models_dir: Path, k: int = 2):
    if not models_dir.exists():
        return []
    candidates = []
    for p in models_dir.iterdir():
        if not p.is_file() or p.suffix != ".pt":
            continue
        m = _EQ_RE.match(p.name)
        if not m:
            continue
        eq = int(m.group("eq"))
        ep = int(m.group("ep"))
        candidates.append((eq, ep, p))
    candidates.sort(key=lambda t: (t[0], t[1]), reverse=True)
    return [t[2].resolve() for t in candidates[:k]]


def ensemble_raw_signals(X_t: torch.Tensor, m_len: int, num_features: int, models_infos, device: str):
    out = []
    for model_info in models_infos:
        arch_ctor = model_info["architecture"]
        for model_path in model_info["paths"]:
            model = arch_ctor(num_features).to(device)
            model.load_state_dict(torch.load(model_path, map_location=torch.device(device)))
            model.eval()
            Y_pred = torch.zeros([m_len, num_features], device=device, dtype=torch.float32)
            with torch.no_grad():
                chunks = 16
                while True:
                    try:
                        mchunk = max(1, m_len // chunks)
                        for j in range(chunks + 1):
                            start = mchunk * j
                            if start >= m_len:
                                break
                            end = min(mchunk * (j + 1), m_len)
                            Y_pred[start:end] = model(X_t[start:end])
                        break
                    except Exception as err:
                        print(f"Chunks ({chunks}) err: {err}")
                        chunks += 2
            out.append(Y_pred.detach().cpu().numpy()[:, 0].copy())
    return out


def trading_backtest(all_raw_signals, opens, highs, lows, closes, phase_label: str, *,
                    taker_fee=0.00055, pos_size=1000.0, sl_points=10.0, tp_points=1000.0, threshold=0.0007):
    cash = float(pos_size)
    position = 0
    entry_price = None

    equities = []
    total_fees = 0.0

    def fee():
        return pos_size * taker_fee

    def realize_to(price):
        nonlocal cash, entry_price, position, total_fees
        if position == 0 or entry_price is None:
            return
        pct = (price - entry_price) / entry_price
        pnl = pos_size * (pct if position == 1 else -pct)
        f = fee()
        cash += pnl - f
        total_fees += f
        entry_price = None
        position = 0

    def mark_to_market(close_price):
        if position == 0 or entry_price is None:
            return cash
        pct = (close_price - entry_price) / entry_price
        return cash + pos_size * (pct if position == 1 else -pct)

    n = len(opens) - 1
    for i in range(n):
        curr_open = opens[i]
        curr_high = highs[i]
        curr_low = lows[i]
        curr_close = closes[i]

        raw_values = [arr[i] for arr in all_raw_signals]
        mean_signal = float(np.mean(raw_values))

        if mean_signal > threshold:
            desired = 1
        elif mean_signal < -threshold:
            desired = -1
        else:
            desired = 0

        if position != 0 and entry_price is not None:
            sl_price = entry_price - sl_points if position == 1 else entry_price + sl_points
            tp_price = entry_price + tp_points if position == 1 else entry_price - tp_points

            if (position == 1 and curr_low <= sl_price) or (position == -1 and curr_high >= sl_price):
                realize_to(sl_price)
            elif (position == 1 and curr_high >= tp_price) or (position == -1 and curr_low <= tp_price):
                realize_to(tp_price)

        if position == 0 and desired != 0:
            cash -= fee()
            entry_price = curr_open
            position = desired

        equities.append(mark_to_market(curr_close))

    if position != 0:
        realize_to(closes[len(equities) - 1])

    print(f"{phase_label}: final equity = {equities[-1]:.2f} | fees = {total_fees:.2f}")
    return equities


def run_pair_holdout_backtest(pair_root: Path, data_csv: Path, *, k_checkpoints: int = 2, seq_len: int = 64,
                             backtest_cfg: dict | None = None, device: str | None = None):
    pair_root = Path(pair_root)
    artifacts = pair_root / "artifacts"

    with open(artifacts / "split_info.json", "r", encoding="utf-8") as f:
        sp = json.load(f)

    with open(artifacts / "scalers.pkl", "rb") as f:
        train_scalers = pickle.load(f)

    df = pd.read_csv(data_csv)

    # Split indices (expects these keys from existing pipeline)
    train_end = int(sp["train_end"])
    val_end = int(sp["val_end"])

    df_sel = df.iloc[:val_end].reset_index(drop=True)
    df_hold = df.iloc[val_end:].reset_index(drop=True)

    X_sel, Y_sel, num_features, sel_opens, sel_highs, sel_lows, sel_closes, _ = preprocess_data(seq_len, df_sel, train_scalers)
    X_hold, Y_hold, _, ho_opens, ho_highs, ho_lows, ho_closes, _ = preprocess_data(seq_len, df_hold, train_scalers)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X_sel_t = torch.from_numpy(X_sel.astype(np.float32)).to(device, dtype=torch.float32)
    X_hold_t = torch.from_numpy(X_hold.astype(np.float32)).to(device, dtype=torch.float32)

    arch_map = {"conv1d": Model_1, "lstm": Model_2, "hybrid": Model_3}
    arch_models_dir = {
        "conv1d": pair_root / "CONV1D_model_training" / "CONV1D_model_training" / "models",
        "lstm": pair_root / "LSTM_model_training" / "LSTM_model_training" / "models",
        "hybrid": pair_root / "hybrid_model_training" / "models",
    }

    models_infos = []
    for arch_name in ["conv1d", "lstm", "hybrid"]:
        models_dir = arch_models_dir[arch_name]
        abs_paths = top_k_checkpoints(models_dir, k=k_checkpoints)
        if not abs_paths:
            raise FileNotFoundError(f"No .pt checkpoints found for {arch_name} in {models_dir}")
        print(f"{pair_root.name}:{arch_name} using {len(abs_paths)} checkpoint(s)")
        for p in abs_paths:
            print(f"  - {p}")
        models_infos.append({"paths": abs_paths, "architecture": arch_map[arch_name]})

    m_sel = len(sel_opens)
    m_hold = len(ho_opens)

    raw_hold = ensemble_raw_signals(X_hold_t, m_hold, num_features, models_infos, device)

    cfg = backtest_cfg or {}
    equities_holdout = trading_backtest(
        raw_hold,
        ho_opens,
        ho_highs,
        ho_lows,
        ho_closes,
        f"{pair_root.name} holdout",
        **cfg,
    )

    return {
        "pair": pair_root.name,
        "equities": equities_holdout,
        "opens": ho_opens,
    }


In [ ]:
# === CONFIG ===
# Escolhe 4 pares (roots) e garante que existe `data/<PAIR>-1h-data.csv` dentro de cada root.
# BTC e ETH já existem; os outros (PAXG/SOL/LINK/XRP) ficam prontos quando tiveres dados + treinos.

ROOT = Path(r"C:\Users\jtoma\projects\BTC - NN bot").resolve()

PAIRS = [
    {
        "name": "BTC",
        "root": ROOT / "CNN",
        "data": ROOT / "CNN" / "data" / "BTCUSDT-1h-data.csv",
        "cfg": {"pos_size": 1000.0, "sl_points": 10.0, "tp_points": 1000.0, "threshold": 0.0007},
    },
    {
        "name": "ETH",
        "root": ROOT / "CNN_ETH",
        "data": ROOT / "CNN_ETH" / "data" / "ETHUSDT-1h-data.csv",
        "cfg": {"pos_size": 1000.0, "sl_points": 10.0, "tp_points": 1000.0, "threshold": 0.0007},
    },
    # Exemplo (ativa quando tiveres dados + models):
    # {
    #     "name": "PAXG",
    #     "root": ROOT / "CNN_PAXG",
    #     "data": ROOT / "CNN_PAXG" / "data" / "PAXGUSDT-1h-data.csv",
    #     "cfg": {"pos_size": 800.0, "sl_points": 5.0, "tp_points": 500.0, "threshold": 0.0007},
    # },
    # {
    #     "name": "SOL",
    #     "root": ROOT / "CNN_SOL",
    #     "data": ROOT / "CNN_SOL" / "data" / "SOLUSDT-1h-data.csv",
    #     "cfg": {"pos_size": 600.0, "sl_points": 20.0, "tp_points": 1200.0, "threshold": 0.0007},
    # },
    # {
    #     "name": "LINK",
    #     "root": ROOT / "CNN_LINK",
    #     "data": ROOT / "CNN_LINK" / "data" / "LINKUSDT-1h-data.csv",
    #     "cfg": {"pos_size": 600.0, "sl_points": 10.0, "tp_points": 800.0, "threshold": 0.0007},
    # },
    # {
    #     "name": "XRP",
    #     "root": ROOT / "CNN_XRP",
    #     "data": ROOT / "CNN_XRP" / "data" / "XRPUSDT-1h-data.csv",
    #     "cfg": {"pos_size": 600.0, "sl_points": 10.0, "tp_points": 800.0, "threshold": 0.0007},
    # },
]

PORTFOLIO_WEIGHTS = None  # None => equal weights; or dict {"BTC":0.25, ...}

set_all_seeds(0)


In [ ]:
# === RUN (per-pair) ===
results = []
for p in PAIRS:
    if not p["root"].exists():
        raise FileNotFoundError(f"Pair root not found: {p['root']}")
    if not p["data"].exists():
        raise FileNotFoundError(f"Data CSV not found: {p['data']}")

    res = run_pair_holdout_backtest(
        p["root"],
        p["data"],
        k_checkpoints=2,
        seq_len=64,
        backtest_cfg=p.get("cfg") or {},
    )

    eq = res["equities"]
    dd = calculate_max_drawdown(eq)
    sh = calculate_sharpe_ratio(eq)
    print(f"{p['name']}: equity={eq[-1]:.2f} | sharpe={sh:.4f} | maxDD={dd['max_drawdown']:.2%}")

    results.append({"name": p["name"], **res, "maxdd": dd, "sharpe": sh})

# === PORTFOLIO MIX ===
# Align by the minimum length (simple + robust)
min_len = min(len(r["equities"]) for r in results)

# Convert each equity to returns, then weight-sum returns
names = [r["name"] for r in results]
if PORTFOLIO_WEIGHTS is None:
    w = {n: 1.0 / len(names) for n in names}
else:
    w = PORTFOLIO_WEIGHTS

rets = {}
for r in results:
    eq = np.array(r["equities"][:min_len], dtype=float)
    rret = (eq[1:] - eq[:-1]) / eq[:-1]
    rets[r["name"]] = rret

portfolio_ret = np.zeros(min_len - 1, dtype=float)
for n in names:
    portfolio_ret += w.get(n, 0.0) * rets[n]

portfolio_equity = [1000.0]
for rr in portfolio_ret:
    portfolio_equity.append(portfolio_equity[-1] * (1.0 + rr))

p_dd = calculate_max_drawdown(portfolio_equity)
p_sh = calculate_sharpe_ratio(portfolio_equity)
print(f"PORTFOLIO: equity={portfolio_equity[-1]:.2f} | sharpe={p_sh:.4f} | maxDD={p_dd['max_drawdown']:.2%}")

plot_equity_simple(portfolio_equity, title=f"Portfolio equity (4 pairs) | DD {p_dd['max_drawdown']:.2%} | Sharpe {p_sh:.4f}")
